# RELIABILITY: TWO-DIMENSIONAL HAT FUNCTION

This example showcases the application of various reliability analysis
methods in UQ[Py]Lab to a two-dimensional hat function.

## 1 - INITIALIZATION
### Package imports

In [ ]:
from uqpylab import sessions
import pprint; pp = pprint.PrettyPrinter(depth=2)

### Start a remote UQCloud session

In [ ]:
# Start the session
mySession = sessions.cloud()
# (Optional) Get a convenient handle to the command line interface
uq = mySession.cli
# Set the timeout
mySession.timeout = 2000
# Reset the session
mySession.reset()


### Set the random seed for reproducibility

In [ ]:
uq.rng(100,'twister');

## 2 - COMPUTATIONAL MODEL
The two-dimensional hat function is defined as follows:
$$g(x_1, x_2) = 20 - (x_1 - x_2)^2 - 8 (x_1 + x_2 - 4)^3$$

### Create a limit state function model based on the hat function using a string, written below in a vectorized operation:


In [ ]:
ModelOpts = { 
    'Type': 'Model', 
    'ModelFun':'Hat.model',
    'isVectorized': True
}
myModel = uq.createModel(ModelOpts)

## 3 - PROBABILISTIC INPUT MODEL
The probabilistic input model consists of two independent and identically-distributed Gaussian random variables:
$$X_i \sim \mathcal{N}(0.25, 1), \quad i = 1, 2$$

### Specify the marginals of the two input random variables:

In [ ]:
InputOpts = {
    "Marginals": [
        {"Name":"X1",
         "Type":"Gaussian",
         "Parameters":[0.25,1]
        },
        {"Name":"X2",
         "Type":"Gaussian",
         "Parameters":[0.25,1]
        }
    ]
}

### Create an INPUT object based on the specified marginals and print its properties

In [ ]:
myInput = uq.createInput(InputOpts)
uq.print(myInput)

## 4 - STRUCTURAL RELIABILITY

Failure event is defined as $g(\mathbf{x}) \leq 0$. The failure probability is then defined as $P_f = P[g(\mathbf{x})\leq 0]$.
Reliability analysis is performed with the following methods:

* Monte Carlo simulation (MCS)
* Subset simulation
* Adaptive-Kriging-Monte-Carlo-Simulation (AK-MCS)
* Adaptive-Polynomial-Chaos-Kriging-Monte-Carlo-Simulation (APCK-MCS)


### 4.1 Monte Carlo simulation (MCS)
Define a `Reliability` analysis using Monte Carlo simulation (MCS) by specifying the maximum sample size, the size of the batchs, and the target coefficient of variation:

In [ ]:
MCSOpts = {
    "Type":"Reliability",
    "Method":"MCS",
    "Simulation": { 
        "MaxSampleSize":1.0E+6,
        "BatchSize":1.0E+5,
        "TargetCoV":0.01
    }
}

Run the Monte Carlo simulation and print out a report of the results:

In [ ]:
MCSAnalysis = uq.createAnalysis(MCSOpts)

Print out a report of the results:

In [ ]:
uq.print(MCSAnalysis)

Create a graphical representation of the results (convergence of $P_f$, convergence of $\beta$, and MCS samples):

In [ ]:
uq.display(MCSAnalysis);

### 4.2 Subset Simulation
Define a `Reliability` analysis using subset simulation by specifying the batch size:

In [ ]:
SubsetSimOpts = {
    "Type":"Reliability",
    "Method":"Subset",
    "Simulation": {
        "BatchSize": 10000
    }
}


Run reliability analysis with Subset simulation:

In [ ]:
SubsetSim = uq.createAnalysis(SubsetSimOpts)

Print out a report of the results:

In [ ]:
uq.print(SubsetSim)

Create a graphical representation of the results (plot the subsets):

In [ ]:
uq.display(SubsetSim);

### 4.3 Adaptive-Kriging-Monte-Carlo-Simulation (AKMCS)
Select the Reliability module and the AK-MCS method:

In [ ]:
AKMCSOpts = {
    "Type": "Reliability",
    "Method":"AKMCS",
    "Simulation": {
        "MaxSampleSize": 1e6    # Specify the size of the Monte Carlo sample set used for the Monte Carlo simulation
    },
    "AKMCS": {
        "MaxAddedED": 20,       # Specify the maximum number of sample points added to the experimental design
        "IExpDesign": {         # Specify the initial experimental design
            "N": 20,  
            "Sampling": "LHS"
        },
        "Kriging": {            # Specify the options for the Kriging metamodel (note that all Kriging options are supported)
            "Corr": {
                "Family": "Gaussian"
                }
        },
        "Convergence": "stopPf",  # Specify the convergence criterion for the adaptive experimental design algorithm (here, it is based on the failure probability estimate):
        "LearningFunction": "EFF" # Specify the learning function (here, it is the expected feasibility function (EFF))
    }
}

Run the AK-MCS simulation:

In [ ]:
AKMCSAnalysis = uq.createAnalysis(AKMCSOpts)

Print out a report of the results:

In [ ]:
uq.print(AKMCSAnalysis)

Create a graphical representation of the results (plot AKMCS convergence and the AKMCS experimental design and the limit state surface):

In [ ]:
uq.display(AKMCSAnalysis);

## 4.4 Adaptive-Polynomial-Chaos-Kriging-Monte-Carlo-Simulation (APCK-MCS)

APCK-MCS is a variation of AK-MCS in which the Kriging model is replaced by a PC-Kriging (PCK) model.

Select the Reliability module and the APCK-MCS method:

In [ ]:
APCKOpts = {
    "Type": "Reliability",
    "Method": "AKMCS",
    "AKMCS": {
        "MetaModel": "PCK",
        "PCK": {
            "Kriging": {
                "Corr": {
                    "Family": "Gaussian"
                }
            }
        },
        "IExpDesign": {
            "N": 5
        }
    },
    "Simulation": {
        "MaxSampleSize": 1.0E+6
    }
}


Run the APCK-MCS simulation:

In [ ]:
myAPCKMCSAnalysis = uq.createAnalysis(APCKOpts)

Print out a report of the results:

In [ ]:
uq.print(myAPCKMCSAnalysis)

Create a graphical representation of the results (Create a graphical representation of the results (plot APCK-MCS convergence and the APCK-MCS experimental design and the limit state surface):

In [ ]:
uq.display(myAPCKMCSAnalysis);

## 4.5 Stochastic spectral embedding-based reliability (SSER)

SSER is an active learning reliability method that constructs a stochastic spectral embedding of the limit state function with smart refinement, partitioning, and sample enrichment strategies.

Select the Reliability module and the SSER method:

In [ ]:
SSEROpts = {
    "Type": "Reliability",
    "Method": "SSER",
    "SSER": {
        "ExpDesign": {
            "NEnrich": 20    # Set the number of samples to be added in every refinement domain
        },
        "ExpOptions": {
            "Degree": [0, 1, 2, 3, 4]   # Select the polynomial degree of the residual expansion
        }
    }
}

Run the SSER analysis:

In [ ]:
mySSERAnalysis = uq.createAnalysis(SSEROpts)

Print out a report of the results:

In [ ]:
uq.print(mySSERAnalysis)

Visualize the results of the analysis:

In [ ]:
uq.display(mySSERAnalysis);

## Terminate the remote UQCloud session:

In [ ]:
mySession.quit()